In [2]:
import requests
from bs4 import BeautifulSoup

# URL of the page to scrape
base_url = "https://reate.cprm.gov.br"
basins_link = "/anp/TERRESTRE"

# Send a GET request to the page
response = requests.get(base_url + basins_link)
basin_data = {}

# Check if the request was successful
if response.status_code == 200:
    # Parse the page content using BeautifulSoup
    soup = BeautifulSoup(response.content, 'html.parser')
        
    # Find all <h4> tags that start with 'Bacia'
    for h4_tag in soup.find_all('h4'):
        if h4_tag.get_text().strip().startswith('Bacia'):
            basin_name = h4_tag.get_text(strip=True)
            
            # Find the nearest <a> tag with an href attribute
            link_tag = h4_tag.find_next('a', href=True)
            if link_tag:
                link = link_tag['href']
                # Store the basin name and link in the dictionary
                basin_data[basin_name] = link

    # Output the extracted basin names and links
    for basin, link in basin_data.items():
        print(f"Basin name: {basin}\nLink: {link}\n")
else:
    print(f"Failed to retrieve the page. Status code: {response.status_code}")

Basin name: Bacia do Acre - Madre de Dios
Link: https://reate.cprm.gov.br/arquivos/index.php/s/1H3Dg7jT4gInGMC

Basin name: Bacia de Alagoas
Link: https://reate.cprm.gov.br/arquivos/index.php/s/UIgVZobfQwyLeA1

Basin name: Bacia Do Amazonas
Link: https://reate.cprm.gov.br/arquivos/index.php/s/IPNA8z7hO1vHsxI

Basin name: Bacia do Araripe
Link: https://reate.cprm.gov.br/arquivos/index.php/s/ygv6FS5D91I6mef

Basin name: Bacia de Barreirinhas
Link: https://reate.cprm.gov.br/arquivos/index.php/s/WXhrH3KTduh9IVM

Basin name: Bacia de Bragança - Vizeu
Link: https://reate.cprm.gov.br/arquivos/index.php/s/Ril60VKtkXRGyE0

Basin name: Bacia do Marajó
Link: https://reate.cprm.gov.br/arquivos/index.php/s/vPEuDaXVoAqLZGx

Basin name: Bacia do Pantanal
Link: https://reate.cprm.gov.br/arquivos/index.php/s/HyVBmvQ8rht4u3k

Basin name: Bacia do Paraná
Link: https://reate.cprm.gov.br/arquivos/index.php/s/z0XoautAuswCSbf

Basin name: Bacia do Parecis - Alto Xingú
Link: https://reate.cprm.gov.br/arquivos

In [3]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import requests
import os
import time

def fetch_by_ID(url: str, id: str) -> str:
    """
    args
        driver: webdriver like firefox.
        url:
        id: 
    returns
        html content
    """
    print(f'url {url}')
    driver = webdriver.Firefox()
    try:
        driver.get(url)

        WebDriverWait(driver, 20).until(
                EC.presence_of_element_located((By.ID, id))
            )
        time.sleep(0.1)
        # Once the element is present, find it and get the HTML content
        tbody = driver.find_element(By.ID, id)
        tbody_html = tbody.get_attribute('innerHTML')
    except Exception as e:
        print(e)
        print(f"failed to open this url {url}")
    finally:
        driver.close()
    return tbody_html


def fetch_table_contents(url: str) -> BeautifulSoup:
    print('fetching table contents...')
    tbody = fetch_by_ID(url, 'fileList')
    tbody_soup = BeautifulSoup(tbody, 'html.parser')
    table_rows = tbody_soup.find_all('tr')
    return table_rows


def fetch_all_folders_names(url: str) -> list:
    print('fetching all folders names...')
    folders_names = []
    table_rows = fetch_table_contents( url)
    for tr in table_rows: 
        td_filename = tr.find('td', class_='filename')
        if td_filename:
            span_innernametext = td_filename.find('span', class_='innernametext').get_text(strip=True)
            folders_names.append(span_innernametext)
    return folders_names


def download_composite_profile(url: str, base_url: str, basin_name: str):
    print('downloading...')
    table_rows = fetch_table_contents(url)
    for tr in table_rows:
        link_tag = tr.find('a', href=True)['href']
        if link_tag:
            response = requests.get(base_url + link_tag)
            if response.status_code == 200:
                directory_path = '../data/' + basin_name + "/"
                # Create the directory if it doesn't exist
                os.makedirs(directory_path, exist_ok=True)
                with open(directory_path + link_tag.split('=')[-1], 'wb') as file:
                    file.write(response.content)


def fetch_composite_profile( urls: list, base_url: str, basin_name: str):
    print('fetching composite profile...')
    copy_urls = urls.copy()
    while len(copy_urls) > 0:
        print(len(copy_urls))
        url = copy_urls.pop()
        folders_names = fetch_all_folders_names(url)
        leaf = 'AGP' in folders_names or 'Perfil Convencional' in folders_names or 'Perfil Composto' in folders_names
        if 'AGP' in folders_names:
            AGP_url = url + str('%2F') + 'AGP'
            download_composite_profile(AGP_url, base_url, basin_name)
        if 'Perfil Convencional' in folders_names:
            conventional_profile_url = url + str('%2F') + 'Perfil%20Convencional'
            download_composite_profile(conventional_profile_url, base_url, basin_name)
        if 'Perfil Composto' in folders_names:
            composite_profile_url = url + str('%2F') + 'Perfil%20Composto'
            download_composite_profile(composite_profile_url, base_url, basin_name)
        if leaf:
            continue
        for folder_name in folders_names:
            new_url = url + '%2F' + folder_name
            copy_urls.append(new_url)





In [4]:
import concurrent.futures
# Path to the WebDriver (you need to have the appropriate WebDriver for your browser installed)
# For example, if you're using Chrome, you need chromedriver installed and its path specified here
base_path = str('?path=%2FPOCO')
# Function that fetches composite profiles
def fetch_composite_profile_task(basin_name, basin_link):
    fetch_composite_profile([basin_link + base_path], base_url, basin_name)
    
# Create a ThreadPoolExecutor with a maximum of 3 threads
with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
    # Submit tasks to the executor
    futures = []
    for basin_name, basin_link in basin_data.items():
        future = executor.submit(fetch_composite_profile_task, basin_name, basin_link)
        futures.append(future)
    
    # Optionally, you can wait for all futures to complete if you need to
    for future in concurrent.futures.as_completed(futures):
        try:
            result = future.result()  # Get the result if needed
        except Exception as exc:
            print(f'Generated an exception: {exc}')

fetching composite profile...
1
fetching all folders names...
fetching table contents...
url https://reate.cprm.gov.br/arquivos/index.php/s/1H3Dg7jT4gInGMC?path=%2FPOCO
fetching composite profile...
1
fetching all folders names...
fetching table contents...
url https://reate.cprm.gov.br/arquivos/index.php/s/UIgVZobfQwyLeA1?path=%2FPOCO
fetching composite profile...
1
fetching all folders names...
fetching table contents...
url https://reate.cprm.gov.br/arquivos/index.php/s/IPNA8z7hO1vHsxI?path=%2FPOCO
4
fetching all folders names...
fetching table contents...
url https://reate.cprm.gov.br/arquivos/index.php/s/IPNA8z7hO1vHsxI?path=%2FPOCO%2FCategoria-9
fetching composite profile...
1
fetching all folders names...
fetching table contents...
url https://reate.cprm.gov.br/arquivos/index.php/s/ygv6FS5D91I6mef?path=%2FPOCO
2
fetching all folders names...
fetching table contents...
url https://reate.cprm.gov.br/arquivos/index.php/s/1H3Dg7jT4gInGMC?path=%2FPOCO%2FCategoria-2
2
fetching all fol

In [ ]:
"https://reate.cprm.gov.br/arquivos/index.php/s/IPNA8z7hO1vHsxI?path=%2FPOCO%2FCategoria-9%2F9-FZ-15-AM%2FPERFIS%2FLIS%2F9FZ__0015__AM_9FZ__0015__AM"